# Kernel Angle Labeling Mini-Programs

Two tools for annotating the main-axis direction of maize kernels.
These labels train the **ResNet axis model** used in the pipeline's measurement stage.

---
## Overview

```
SAM2 mask (binary)  +  kernel crop (RGB)
        │
        ▼
  ┌──────────────────────────┐
  │ 1. centroid_overlay.py   │
  │    Compute mask centroid  │
  │    via image moments      │
  │    → centroids.csv        │
  └──────────────────────────┘
        │
        ▼
  ┌──────────────────────────┐
  │ 2. angle_labeler.py      │
  │    Interactive GUI:       │
  │    adjust red arrow to    │
  │    match kernel axis      │
  │    → angle_labels.csv     │
  └──────────────────────────┘
        │
        ▼
  ResNet training data
  (kernel crop → cosθ, sinθ)
```

### Why two angles are saved

| Pair | Meaning | Axis type |
|------|---------|----------|
| `cosθ, sinθ` | Directed axis (tip → crown) | Primary — the red arrow direction |
| `cos2θ, sin2θ` | Undirected axis (orientation) | Auxiliary — same for θ and θ+180° |

The directed axis is what the ResNet model is trained to predict.
The undirected pair `cos2θ, sin2θ` is orientation-agnostic — useful when
the tip/crown distinction is ambiguous.

---
## Tool 1: centroid_overlay.py

Batch utility. Reads SAM binary masks, computes the **geometric centroid**
via image moments (`cv2.moments`), draws it on the RGB kernel crop,
and saves a CSV.

In [ ]:
# Run centroid_overlay.py
!python centroid_overlay.py \
    --images /path/to/subimages/ \
    --masks  /path/to/masks_binary/ \
    --out-csv centroids.csv \
    --overlay-dir centroid_overlay/

**What it does step-by-step:**

1. Reads every `.png` / `.jpg` from `--images`
2. Finds the matching binary mask in `--masks` (matches by filename)
3. Thresholds the mask at 127 → keeps only the largest connected component
4. Computes centroid:  
   `cx = M10 / M00`  `cy = M01 / M00`
5. Draws a yellow dot + yellow contour on the RGB crop
6. Saves the overlay image to `--overlay-dir`
7. Writes `centroids.csv` with columns:

| Column | Meaning |
|--------|--------|
| image_path | Full path to RGB crop |
| mask_path | Full path to binary mask |
| status | `ok`, `mask_missing`, `moments_failed` |
| center_x | Centroid x-coordinate (float) |
| center_y | Centroid y-coordinate (float) |

**When to use:** Before running `angle_labeler.py` — the centroid serves
as the rotation pivot point for the axis arrow.

**Output preview:** Yellow dot at kernel center + yellow contour outline.

---
## Tool 2: angle_labeler.py

Interactive GUI for manually adjusting the kernel main-axis direction.
Uses OpenCV `imshow` + keyboard input — no web UI needed.

In [ ]:
# Run angle_labeler.py
!python angle_labeler.py \
    --images /path/to/subimages/ \
    --masks  /path/to/masks_binary/ \
    --centroids centroids.csv \
    --out-csv angle_labels_directed.csv \
    --overlay-dir angle_overlay/ \
    --display-size 640            # optional, default 640px
    --angle-step 1                # keyboard rotation step in degrees (default 1)
    --resume angle_labels_directed.csv  # optional: continue from a previous CSV

### How the GUI Works

```
┌──────────────────────────────────────┐
│  Status bar (top 56px):              │
│  θ=127.3°  center=(245,198)         │
│  cosθ=-0.607  sinθ=0.795             │
│  [IMG_14_kernel_001]  (12/350)      │
├──────────────────────────────────────┤
│                                      │
│          /\  ← red arrow             │
│         /  \    (axis direction)     │
│        /    \                        │
│       /  ●   \    ← yellow centroid  │
│      /        \                      │
│     /          \                     │
│    /   kernel   \                    │
│   /   outline    \                   │
│                                      │
└──────────────────────────────────────┘
```

### Keyboard Controls

| Key | Action |
|-----|--------|
| **Q** / **S** | Rotate arrow ← left / right → (step = `--angle-step` degrees) |
| **Up / Down** | Fine rotation (±0.1° per press) |
| **Left / Right** | Fine rotation (±0.1° per press) |
| **Mouse drag** | Drag the arrow tip to set angle directly |
| **Space / Enter** | ✅ Confirm — save label, advance to next image |
| **Delete / Backspace** | ⏭️ Skip this kernel, advance to next |
| **ESC** | 💾 Save CSV and exit (can resume later with `--resume`) |

### What Gets Saved

Each confirmed kernel writes a row to `angle_labels_directed.csv`:

| Column | Meaning |
|--------|--------|
| image_label | Kernel crop filename (e.g. `IMG_14_kernel_001.jpg`) |
| status | `labeled` (confirmed) or `skipped` (deleted) |
| theta_deg | Axis angle in degrees (0-360) |
| theta_rad | Axis angle in radians |
| cos_theta | cos(θ) — directed axis x-component |
| sin_theta | sin(θ) — directed axis y-component |
| cos2theta | cos(2θ) — undirected axis (orientation only) |
| sin2theta | sin(2θ) — undirected axis |
| center_x / center_y | Centroid coordinates |
| axis_x1 / axis_y1 | Axis start point (centroid) |
| axis_x2 / axis_y2 | Axis end point (contour intersection) |

### Resuming a Session

```bash
# If you close the GUI (ESC) or it crashes:
python angle_labeler.py \
    --images /path/to/subimages/ \
    --masks  /path/to/masks_binary/ \
    --centroids centroids.csv \
    --resume angle_labels_directed.csv \
    --overlay-dir angle_overlay/
```

Already-labeled images are **skipped automatically**. You pick up
right where you left off.

---
## Workflow Summary

```bash
# Step 1: Compute centroids from SAM masks
python centroid_overlay.py \
    --images subimages/ \
    --masks masks_binary/ \
    --out-csv centroids.csv \
    --overlay-dir centroid_overlay/

# Step 2: Interactively label axis angles
python angle_labeler.py \
    --images subimages/ \
    --masks masks_binary/ \
    --centroids centroids.csv \
    --out-csv angle_labels_directed.csv \
    --overlay-dir angle_overlay/

# Step 3: Train ResNet (in resnet/ directory)
python resnet/train_resnet_angle.py resnet/config.yaml
# Input:  kernel RGB crops + angle_labels_directed.csv
# Output: resnet/weights/best.pt
```

The trained ResNet model predicts `cosθ, sinθ` directly from a kernel crop
and is used in the pipeline's `kernel_metrics.py` stage 5.